# SkinMatch: run the full scrape
Put `scrape_skincare.py` and this notebook in the same folder. Run the cells **in order, one at a time**.
Read each site's Terms of Use before scraping it.

## 1. Install (one time)

In [13]:
!pip install requests beautifulsoup4 lxml pandas

In [14]:
# Only needed for Jenpharm (its ingredients load via JavaScript). Skip if you don't need Jenpharm.
!pip install playwright
!playwright install chromium

## 2. Settings
Delete a brand from the list to skip it. Raise `MAX_PER_BRAND` for more products.

In [15]:
ORDER = ["The Ordinary", "CeraVe", "Cetaphil", "Blissful"]
MAX_PER_BRAND = 300
MAX_PAGES_PER_BRAND = 400

## 3. Scrape
Safe to stop (press the stop button) and run again: saved products are skipped.

In [16]:
import pandas as pd
import importlib
import scrape_skincare as s
importlib.reload(s)          # picks up a newly replaced scrape_skincare.py

s.MAX_PRODUCTS_PER_BRAND = MAX_PER_BRAND
s.MAX_PAGES_FETCHED_PER_BRAND = MAX_PAGES_PER_BRAND
s.migrate_csv()

results = []
for brand in ORDER:
    if brand not in s.SITES:
        continue
    try:
        done = s.load_done_urls()
        results.append(s.scrape_brand(brand, s.SITES[brand], done))
    except KeyboardInterrupt:
        print("Stopped. Run this cell again to continue.")
        break
    except Exception as e:
        print(f"!! {brand} failed: {type(e).__name__}: {e}")
        results.append({"brand": brand, "error": str(e)[:100]})

print("\n========== SUMMARY ==========")
for r in results:
    print(r)


=== The Ordinary ===
  reminder: read this site's Terms of Use before scraping (robots.txt is not the whole story)
  sitemap gave no product URLs; crawling listing pages instead
  candidate URLs (allowed, not yet scraped): 0
  nothing to do (paste manual_urls if the sitemap gave nothing)

=== CeraVe ===
  reminder: read this site's Terms of Use before scraping (robots.txt is not the whole story)
  candidate URLs (allowed, not yet scraped): 13
    not a product page: https://www.cerave.com/skincare/moisturizers/body-moisturizers
    not a product page: https://www.cerave.com/skincare/cleansers/cream-facial-cleansers
    not a product page: https://www.cerave.com/skincare/moisturizers/facial-moisturizers
    not a product page: https://www.cerave.com/skincare/cleansers/facial-cleansers-for-oily-skin
    not a product page: https://www.cerave.com/skincare/cleansers/facial-cleansers-for-sensitive-skin-
    not a product page: https://www.cerave.com/skincare/cleansers/facial-cleansers-for-

In [17]:
import pandas as pd

KAGGLE_IN = "data/raw/cosmetics.csv"      # the original Kaggle file, never overwritten
OUT       = "data/combined_products.csv"  # merged output goes to a different file

k = pd.read_csv(KAGGLE_IN)
k.columns = [c.strip().lower() for c in k.columns]
print("Kaggle columns:", k.columns.tolist())   # should include label, brand, name, ingredients, dry, oily ...

def skin_text(r):
    return "|".join(n for n in ["combination", "dry", "normal", "oily", "sensitive"] if r.get(n) == 1)

ext = pd.DataFrame({
    "brand": k["brand"], "name": k["name"], "category": k["label"],
    "ingredients": k["ingredients"].str.lower().str.strip(),
    "skin_types": k.apply(skin_text, axis=1).replace("", pd.NA),
    "price_usd": k["price"], "rating": k.get("rank"),
    "currency": "USD", "source": "kaggle (sephora)"})

# drop placeholder / non-list ingredient text ("No Info", "Visit the boutique...")
ok = ext.ingredients.fillna("").str.count(",") >= 3
ext = ext[ok].copy()
ext["ingredient_count"] = ext.ingredients.str.count(",") + 1

mine = pd.read_csv("skincare_products.csv")
combined = pd.concat([mine, ext], ignore_index=True)

# remove duplicates (same brand + name in both sources; keeps your scraped row)
key = combined.brand.str.lower().str.strip() + "|" + combined.name.str.lower().str.strip()
combined = combined[~key.duplicated(keep="first")]

combined.to_csv(OUT, index=False)
print(len(combined), "products;", combined.skin_types.notna().sum(), "with skin type")
print(combined.source.value_counts())

Kaggle columns: ['label', 'brand', 'name', 'price', 'rank', 'ingredients', 'combination', 'dry', 'normal', 'oily', 'sensitive']
1495 products; 1096 with skin type
source
kaggle (sephora)    1259
brand site           223
shopify shop          13
Name: count, dtype: int64


## 4. Check the data (gate: 300+ products, 200+ with skin type)

In [18]:
df = pd.read_csv("combined.csv")
print(len(df), "products;", df.skin_types.notna().sum(), "with skin type;", (df.ingredient_count > 0).sum(), "with ingredients")
print(df.brand.value_counts())
df.head()

1694 products; 1198 with skin type; 1643 with ingredients
brand
The Ordinary          94
CLINIQUE              79
SEPHORA COLLECTION    66
CeraVe                65
Cetaphil              63
                      ..
TOM FORD               1
KAPLAN MD              1
BLACK UP               1
URBAN DECAY            1
DERMAFLASH             1
Name: count, Length: 119, dtype: int64


,brand,name,category,ingredients,ingredient_count,skin_types,helps_with,price_usd,rating,review_count,url,currency,source
0,The Ordinary,Multi-Peptide Lash and Brow Serum,Lash & Brow Treatments,"aqua/water/eau, glycerin, butylene glycol, myr...",32,all,Hair,17.90,NaN,NaN,https://theordinary.com/en-pk/multi-peptide-la...,USD,brand site
1,The Ordinary,Balancing & Clarifying Serum,Serums,"water, lactococcus ferment lysate, propanediol...",24,all,Signs of Congestion; Dryness; Textural Irregul...,23.90,NaN,NaN,https://theordinary.com/en-pk/balancing-clarif...,USD,brand site
2,The Ordinary,GF 15% Solution,Serums,"aqua/water/eau, butylene glycol, nicotiana ben...",12,all,Advanced Signs of Aging,18.60,NaN,NaN,https://theordinary.com/en-pk/gf-15-solution-g...,USD,brand site
3,The Ordinary,100% Cold-Pressed Virgin Marula Oil,Face Oils,NaN,0,all,Dryness; Antioxidant Support; Hair,13.70,NaN,NaN,https://theordinary.com/en-pk/100-cold-pressed...,USD,brand site
4,The Ordinary,Multi-Antioxidant Radiance Serum,Multi-Active,"aqua/water/eau, propanediol, ascorbyl glucosid...",21,all,Dullness; Antioxidant Support,16.80,NaN,NaN,https://theordinary.com/en-pk/multi-antioxidan...,USD,brand site


## 5. Find bad ingredient rows (marketing text instead of ingredients)

In [19]:
bad = df.ingredients.fillna("").str.split(",").apply(
    lambda p: len(p) > 0 and sum(len(x.split()) > 6 for x in p) > 0.15 * len(p))
print(bad.sum(), "suspicious rows")
df[bad][["brand", "name", "ingredient_count"]]

61 suspicious rows


,brand,name,ingredient_count
216,Cetaphil,Deep Cleansing Bar 3PK,6
229,DRUNK ELEPHANT,Virgin Marula Luxury Facial Oil,1
233,KIEHL'S SINCE 1851,Midnight Recovery Concentrate,11
248,DRUNK ELEPHANT,Virgin Marula Luxury Facial Oil Mini,1
255,JOSIE MARAN,100 percent Pure Argan Oil,1
...,...,...,...
1626,ILIA,Radiant Translucent Powder SPF 20,1
1646,KAPLAN MD,Daily Moisture SPF 30 Concentrate,1
1657,ILIA,Flow-Thru Radiant Transclucent Powder SPF 20,2
1682,SUPERGOOP!,Perfect Day 2-in-1 Everywear Lotion Broad Spec...,4


In [21]:
# Optional: remove the bad rows, then run section 3 again to refetch them
# df[~bad].to_csv("skincare_products.csv", index=False)
import os
p = os.path.abspath("data/combined_filled.csv")
print(p, "->", "exists" if os.path.exists(p) else "NOT FOUND")

C:\Users\HP\Downloads\data\combined_filled.csv -> NOT FOUND


In [25]:
import os, re
import pandas as pd

SRC = "combined.csv"                          # your merged file in Downloads
OUT = os.path.abspath("combined_filled.csv")

ING_RULES = {
    "Signs of Congestion":     r"salicylic acid|benzoyl peroxide|sulfur|tea tree|azelaic acid|zinc pca|willow bark",
    "Signs of Aging":          r"retinol|retinal|retinyl|bakuchiol|peptide|palmitoyl|acetyl hexapeptide|matrixyl|adenosine",
    "Dryness":                 r"ceramide|shea butter|squalane|petrolatum|urea|cholesterol",
    "Dehydration":             r"hyaluronic acid|sodium hyaluronate|polyglutamic",
    "Dullness":                r"ascorbic acid|ascorbyl|vitamin c|glycolic acid|lactic acid|licorice|glycyrrhiza",
    "Uneven Skin Tone":        r"niacinamide|tranexamic|alpha[- ]arbutin|arbutin|kojic|ascorbyl glucoside",
    "Textural Irregularities": r"glycolic acid|lactic acid|mandelic acid|gluconolactone|salicylic acid|papain|bromelain",
    "Look of Redness":         r"centella|madecassoside|allantoin|bisabolol|colloidal oatmeal|avena sativa|aloe barbadensis",
    "Antioxidant Support":     r"tocopherol|resveratrol|ferulic|green tea|camellia sinensis|ubiquinone|coenzyme",
    "Barrier Support":         r"ceramide|cholesterol|phytosphingosine|fatty acid",
    "Oil Control":             r"kaolin|bentonite|zinc pca|charcoal",
}
NAME_RULES = {
    "Cleansing":                r"cleanser|cleansing|face wash|foam|micellar|makeup remov",
    "Sun Protection":           r"\bspf\b|sunscreen|sun protect|uv",
    "Signs of Aging":           r"anti[- ]?aging|age|wrinkle|firm|lift|retinol|youth|repair",
    "Dryness":                  r"moistur|hydrat|cream|balm|rich|nourish",
    "Dullness":                 r"brighten|radiance|glow|vitamin c|illuminat",
    "Signs of Congestion":      r"acne|blemish|pore|clarif|purif|detox|clear",
    "Textural Irregularities":  r"exfoli|peel|resurfac|polish|scrub|aha|bha",
    "Look of Redness":          r"calm|sooth|redness|sensitive|relief",
    "Puffiness & Dark Circles": r"eye",
    "Oil Control":              r"matte|mattif|oil[- ]control|oil[- ]free",
}

def guess_helps(row):
    text = f"{row['name']} {row['category']}".lower()
    ing = str(row["ingredients"]).lower() if pd.notna(row["ingredients"]) else ""
    top = ", ".join(ing.split(", ")[:20])
    found = [c for c, p in NAME_RULES.items() if re.search(p, text)]
    found += [c for c, p in ING_RULES.items() if c not in found and re.search(p, top)]
    return "; ".join(found[:4]) or pd.NA

df = pd.read_csv(SRC)
empty = df["helps_with"].isna() | (df["helps_with"].astype(str).str.strip() == "")
df["helps_with_source"] = "site"
df.loc[empty, "helps_with"] = df[empty].apply(guess_helps, axis=1)
df.loc[empty, "helps_with_source"] = "derived"
df.loc[df["helps_with"].isna(), "helps_with_source"] = pd.NA
df.to_csv(OUT, index=False)

print("filled:", (empty & df.helps_with.notna()).sum(), "of", empty.sum(), "| still empty:", df.helps_with.isna().sum())
print("Saved to:", OUT, "| exists:", os.path.exists(OUT))

filled: 1495 of 1539 | still empty: 44
Saved to: C:\Users\HP\Downloads\combined_filled.csv | exists: True


In [24]:
import os, glob, importlib
import scrape_skincare as s
importlib.reload(s)

print("Notebook folder:", os.getcwd())

# find the merged CSV (the one with the Kaggle rows in it)
candidates = glob.glob("**/combined*.csv", recursive=True)
print("Found:", candidates)

src = next((c for c in candidates if "filled" not in c), None)
if src is None:
    print("No combined CSV found. Run the Kaggle merge cell first.")
else:
    out = os.path.abspath("combined_filled.csv")
    s.fill_helps_with(src, out)
    print("Saved to:", out, "| exists:", os.path.exists(out))

Notebook folder: C:\Users\HP\Downloads
Found: ['combined.csv', 'data\\combined_products.csv', 'data\\raw\\combined.csv']


AttributeError: module 'scrape_skincare' has no attribute 'fill_helps_with'